In [32]:
import pandas as pd
import glob
import os


datasets = ['tax', 'hospital', 'restaurant', 'sisu']

In [33]:

# Read all classification CSV files
csv_files = glob.glob('classification_csv_results/llm_classification_*.csv')

# Load all dataframes
dfs = []
for csv_file in csv_files:
    if not any(dataset in csv_file for dataset in datasets):
        continue
    df = pd.read_csv(csv_file)
    dfs.append(df)

# Combine all dataframes
df_combined = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {len(df_combined)}")
print(f"Datasets: {df_combined['dataset'].unique()}")
print(f"Models: {df_combined['model'].unique()}")
print(f"Prompt keys: {df_combined['prompt_key'].unique()}")
print()

# Simplify model names for display
model_display_names = {
    'gpt-5-mini': 'GPT-5-mini',
    'gpt-5': 'GPT-5',
    'mistral-small-2506': 'Mistral-Small',
    'mistral-large-2512': 'Mistral-Large',
    'llama-3.3-70b-versatile': 'Llama-3.3-70b'
}

# Prompt key display names
prompt_display_names = {
    'zero_shot': 'Zero-shot',
    'zero_shot_with_table': 'Zero-shot',
    'with_examples': 'Few-shot',
    'with_examples_and_table': 'Few-shot'
}

# Add readable columns
df_combined['model_display'] = df_combined['model'].map(model_display_names)
df_combined['prompt_display'] = df_combined['prompt_key'].map(prompt_display_names)
df_combined['context'] = df_combined['use_context'].map({True: 'Yes', False: 'No'})

print("\n" + "="*100)
print("RANKING REPORT: BEST CONFIGURATIONS PER DATASET")
print("Showing configurations with MAXIMUM RECALL, ranked by F1 score")
print("="*100)

# Analyze each dataset
for dataset in sorted(df_combined['dataset'].unique()):
    df_dataset = df_combined[df_combined['dataset'] == dataset].copy()
    
    # Find maximum recall for this dataset
    max_recall = df_dataset['recall'].max()
    
    # Filter to only configurations that achieved max recall
    df_max_recall = df_dataset[df_dataset['recall'] == max_recall].copy()
    
    # Sort by F1 score (descending)
    df_max_recall = df_max_recall.sort_values('f1_score', ascending=False)
    
    print(f"\n📊 {dataset.upper()} DATASET (Max Recall = {max_recall:.4f})")
    print("-" * 100)
    
    # Create a clean display dataframe
    display_df = df_max_recall[['model_display', 'prompt_display', 'context', 'precision', 'recall', 'f1_score']].copy()
    display_df.columns = ['Model', 'Prompt', 'Context', 'Precision', 'Recall', 'F1']
    display_df = display_df.reset_index(drop=True)
    display_df.index = display_df.index + 1  # Start ranking from 1
    display_df.index.name = 'Rank'
    
    print(display_df.to_string())
    print()
    
    # Show top 3 summary
    print(f"🏆 Top 3 for {dataset}:")
    for i, (idx, row) in enumerate(display_df.head(3).iterrows(), 1):
        print(f"  {i}. {row['Model']} | {row['Prompt']} | Context: {row['Context']} | F1: {row['F1']:.4f}")
    print()

print("="*100)


Total rows: 88
Datasets: <ArrowStringArray>
['sisu', 'hospital', 'restaurant', 'tax']
Length: 4, dtype: str
Models: <ArrowStringArray>
[     'mistral-small-2506',              'gpt-5-mini',
                   'gpt-5',      'mistral-large-2512',
 'llama-3.3-70b-versatile',            'sabiazinho-4',
                 'sabia-4']
Length: 7, dtype: str
Prompt keys: <ArrowStringArray>
[                 'zero_shot',       'zero_shot_with_table',
              'with_examples',    'with_examples_and_table',
               'zero_shot_pt',    'zero_shot_with_table_pt',
           'with_examples_pt', 'with_examples_and_table_pt']
Length: 8, dtype: str


RANKING REPORT: BEST CONFIGURATIONS PER DATASET
Showing configurations with MAXIMUM RECALL, ranked by F1 score

📊 HOSPITAL DATASET (Max Recall = 1.0000)
----------------------------------------------------------------------------------------------------
              Model     Prompt Context  Precision  Recall      F1
Rank                          

In [34]:
# CROSS-DATASET ANALYSIS: Best configuration across all datasets
print("\n" + "="*100)
print("🌟 CROSS-DATASET RANKING: BEST OVERALL CONFIGURATIONS")
print("Showing configurations with MAXIMUM AVERAGE RECALL across all datasets, ranked by Avg F1")
print("="*100)

# Group by configuration (model, prompt_key, use_context)
grouped = df_combined.groupby(['model', 'prompt_key', 'use_context']).agg({
    'recall': 'mean',
    'precision': 'mean',
    'f1_score': 'mean'
}).reset_index()

# Rename columns to be clear they are averages
grouped.columns = ['model', 'prompt_key', 'use_context', 'avg_recall', 'avg_precision', 'avg_f1']

# Add display names
grouped['model_display'] = grouped['model'].map(model_display_names)
grouped['prompt_display'] = grouped['prompt_key'].map(prompt_display_names)
grouped['context'] = grouped['use_context'].map({True: 'Yes', False: 'No'})

# Find maximum average recall
max_avg_recall = grouped['avg_recall'].max()

print(f"\n📈 Maximum Average Recall: {max_avg_recall:.4f}")

# Filter to configurations with maximum average recall
best_configs = grouped[grouped['avg_recall'] == max_avg_recall].copy()

# Sort by average F1 score (descending)
best_configs = best_configs.sort_values('avg_f1', ascending=False).reset_index(drop=True)

print(f"Found {len(best_configs)} configuration(s) achieving maximum average recall\n")

# Create clean display dataframe
display_cross = best_configs[['model_display', 'prompt_display', 'context', 'avg_precision', 'avg_f1', 'avg_recall']].copy()
display_cross.columns = ['Model', 'Prompt', 'Context', 'Avg Precision', 'Avg F1', 'Avg Recall']
display_cross = display_cross.reset_index(drop=True)
display_cross.index = display_cross.index + 1  # Start ranking from 1
display_cross.index.name = 'Rank'

print(display_cross.to_string())
print()

# Show the winner
if len(best_configs) > 0:
    winner = best_configs.iloc[0]
    print("="*100)
    print("🏆 BEST OVERALL CONFIGURATION:")
    print("="*100)
    print(f"  Model:     {winner['model_display']}")
    print(f"  Prompt:    {winner['prompt_display']}")
    print(f"  Context:   {winner['context']}")
    print(f"  Avg Recall:    {winner['avg_recall']:.4f}")
    print(f"  Avg Precision: {winner['avg_precision']:.4f}")
    print(f"  Avg F1:        {winner['avg_f1']:.4f}")
    print("="*100)

# Show detailed breakdown per dataset for the winner
print(f"\n📋 Detailed Performance of Best Configuration on Each Dataset:")
print("-" * 100)

winner_config = best_configs.iloc[0]
winner_details = df_combined[
    (df_combined['model'] == winner_config['model']) &
    (df_combined['prompt_key'] == winner_config['prompt_key']) &
    (df_combined['use_context'] == winner_config['use_context'])
].copy()

winner_details = winner_details[['dataset', 'precision', 'recall', 'f1_score']].copy()
winner_details.columns = ['Dataset', 'Precision', 'Recall', 'F1']
winner_details = winner_details.sort_values('Dataset')
winner_details = winner_details.reset_index(drop=True)
winner_details.index = winner_details.index + 1
print(winner_details.to_string())
print()

# Summary statistics
print("\n📊 Summary Statistics for Winner:")
print(f"  Best Dataset (F1):  {winner_details.loc[winner_details['F1'].idxmax(), 'Dataset']} (F1: {winner_details['F1'].max():.4f})")
print(f"  Worst Dataset (F1): {winner_details.loc[winner_details['F1'].idxmin(), 'Dataset']} (F1: {winner_details['F1'].min():.4f})")
print(f"  F1 Range: {winner_details['F1'].min():.4f} - {winner_details['F1'].max():.4f}")
print(f"  F1 Std Dev: {winner_details['F1'].std():.4f}")
print("\n" + "="*100)


🌟 CROSS-DATASET RANKING: BEST OVERALL CONFIGURATIONS
Showing configurations with MAXIMUM AVERAGE RECALL across all datasets, ranked by Avg F1

📈 Maximum Average Recall: 1.0000
Found 2 configuration(s) achieving maximum average recall

           Model     Prompt Context  Avg Precision    Avg F1  Avg Recall
Rank                                                                    
1     GPT-5-mini   Few-shot      No       0.460025  0.615625         1.0
2     GPT-5-mini  Zero-shot      No       0.407775  0.567775         1.0

🏆 BEST OVERALL CONFIGURATION:
  Model:     GPT-5-mini
  Prompt:    Few-shot
  Context:   No
  Avg Recall:    1.0000
  Avg Precision: 0.4600
  Avg F1:        0.6156

📋 Detailed Performance of Best Configuration on Each Dataset:
----------------------------------------------------------------------------------------------------
      Dataset  Precision  Recall      F1
1    hospital     0.5000     1.0  0.6667
2  restaurant     0.6923     1.0  0.8182
3        sisu     0.

In [35]:
# OPTIONAL: Compare top configurations side-by-side
print("\n" + "="*100)
print("📊 COMPARISON: Top 5 Configurations Across Datasets")
print("="*100)

# Get top 5 by average F1 among those with max recall
top_5 = best_configs.head(5)

comparison_data = []
for idx, config in top_5.iterrows():
    config_details = df_combined[
        (df_combined['model'] == config['model']) &
        (df_combined['prompt_key'] == config['prompt_key']) &
        (df_combined['use_context'] == config['use_context'])
    ].copy()
    
    for _, row in config_details.iterrows():
        comparison_data.append({
            'Configuration': f"{config['model_display']} | {config['prompt_display']} | Ctx:{config['context']}",
            'Rank': idx + 1,
            'Dataset': row['dataset'],
            'Precision': row['precision'],
            'Recall': row['recall'],
            'F1': row['f1_score']
        })

comparison_df = pd.DataFrame(comparison_data)

# Pivot to show datasets as columns
for rank in sorted(comparison_df['Rank'].unique()):
    rank_data = comparison_df[comparison_df['Rank'] == rank]
    config_name = rank_data['Configuration'].iloc[0]
    
    print(f"\n#{rank}: {config_name}")
    print("-" * 100)
    
    pivot = rank_data.pivot_table(index='Dataset', values=['Precision', 'Recall', 'F1'], aggfunc='first')
    print(pivot.to_string())
    
    avg_f1 = rank_data['F1'].mean()
    print(f"  → Average F1: {avg_f1:.4f}")

print("\n" + "="*100)


📊 COMPARISON: Top 5 Configurations Across Datasets

#1: GPT-5-mini | Few-shot | Ctx:No
----------------------------------------------------------------------------------------------------
                F1  Precision  Recall
Dataset                              
hospital    0.6667     0.5000     1.0
restaurant  0.8182     0.6923     1.0
sisu        0.5161     0.3478     1.0
tax         0.4615     0.3000     1.0
  → Average F1: 0.6156

#2: GPT-5-mini | Zero-shot | Ctx:No
----------------------------------------------------------------------------------------------------
                F1  Precision  Recall
Dataset                              
hospital    0.6667     0.5000     1.0
restaurant  0.7200     0.5625     1.0
sisu        0.4706     0.3077     1.0
tax         0.4138     0.2609     1.0
  → Average F1: 0.5678

